# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nooragab/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

#Setup

In [2]:
%pip install -q duckdb

import duckdb, os
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("DuckDB connected, HF secret set. Ready to query the warehouse.")

DuckDB connected, HF secret set. Ready to query the warehouse.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal Check 1: Staleness (behind FlyRank's refresh flags)

Checking whether pages that haven't been updated in a while actually show more decline —
the same signal behind `stale_visible_page` / refresh flags from the session.

In [3]:
signal1 = con.sql(f"""
    WITH monthly AS (
        SELECT
            f.content_hash_id,
            SUM(f.gsc_impressions) AS impressions_month,
            SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS imp_second_half
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
        WHERE f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id
    )
    SELECT
        CASE
            WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 30 THEN '0-29d (fresh)'
            WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 90 THEN '30-89d'
            WHEN DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') < 180 THEN '90-179d'
            ELSE '180d+ (stale)'
        END AS staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(CASE WHEN m.imp_second_half < m.imp_first_half THEN 1.0 ELSE 0.0 END), 3) AS decline_rate
    FROM monthly m
    JOIN read_parquet('{rel}/dim_content.parquet') d ON m.content_hash_id = d.content_hash_id
    WHERE m.impressions_month > 0
    GROUP BY staleness_bucket
    ORDER BY staleness_bucket
""").df()
signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,decline_rate
0,0-29d (fresh),149456,0.354
1,180d+ (stale),261,0.494
2,30-89d,25696,0.507
3,90-179d,1325,0.414


**Verdict: MIXED**

Fresh pages (0-29d) show a clearly lower decline rate (35.4%) than all older buckets, which
supports using staleness as a signal. However, the relationship isn't cleanly monotonic —
decline rates bounce between 41-51% across the 30-89d, 90-179d, and 180d+ buckets rather than
rising steadily with age. The two oldest buckets also have far smaller n (1,325 and 261) than
the younger ones (149,456 and 25,696), so those specific rates are less reliable. I'll still
use staleness in the rule — the fresh-vs-not-fresh split is real — but I won't treat "more
stale = more decline" as a precise gradient.

## Signal Check 2: CTR vs. Position Tier (behind the CTR-fix logic)

Checking whether CTR really drops as position gets worse — the signal behind FlyRank's
CTR-fix flag logic, and the same "CTR cliff" from notebook 01's Discovery B.

In [4]:
signal2 = con.sql(f"""
    WITH monthly AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impressions_month,
            SUM(gsc_clicks) AS clicks_month,
            AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT
        CASE
            WHEN avg_position_month <= 3 THEN '1_top_3'
            WHEN avg_position_month <= 10 THEN '2_striking_4_10'
            WHEN avg_position_month <= 20 THEN '3_page_2'
            ELSE '4_deep_20plus'
        END AS position_tier,
        COUNT(*) AS n,
        ROUND(AVG(clicks_month / NULLIF(impressions_month, 0)), 4) AS avg_ctr
    FROM monthly
    WHERE impressions_month >= 100
    GROUP BY position_tier
    ORDER BY position_tier
""").df()
signal2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_tier,n,avg_ctr
0,1_top_3,9031,0.0036
1,2_striking_4_10,46864,0.0032
2,3_page_2,21474,0.0024
3,4_deep_20plus,24072,0.0012


**Verdict: CONFIRMED**

CTR falls in a clean, monotonic pattern as position tier gets worse: 0.36% (top_3) → 0.32%
(striking_4_10) → 0.24% (page_2) → 0.12% (deep_20plus) — roughly a 3x drop from best to worst
tier. Every tier has a large sample size (9,031 to 46,864 rows), so this isn't a small-sample
fluke. This confirms the same "CTR cliff" seen in the starter dataset's Discovery B, now on
the real warehouse data — it's a solid signal to build the rule on.

## My Rule

**In plain words:** flag a page for review if it is stale (not updated in 90+ days) AND has
a meaningful CTR gap below what its position tier normally gets. The score weighs impressions
(so high-traffic pages surface first), whether the page is stale, and the size of its CTR gap.

**Reason code (one):** `stale_ctr_underperformer`

**Action label:** `review_for_refresh`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import os

queue = con.sql(f"""
    WITH monthly AS (
        SELECT
            content_hash_id, client_hash_id,
            SUM(gsc_impressions) AS impressions_month,
            SUM(gsc_clicks) AS clicks_month,
            AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    ),
    tiered AS (
        SELECT *,
            clicks_month / NULLIF(impressions_month, 0) AS ctr_month,
            CASE
                WHEN avg_position_month <= 3 THEN '1_top_3'
                WHEN avg_position_month <= 10 THEN '2_striking_4_10'
                WHEN avg_position_month <= 20 THEN '3_page_2'
                ELSE '4_deep_20plus'
            END AS position_tier
        FROM monthly
        WHERE impressions_month >= 100
    ),
    tier_avg AS (
        SELECT position_tier, AVG(ctr_month) AS tier_avg_ctr
        FROM tiered GROUP BY position_tier
    )
    SELECT
        t.content_hash_id, t.client_hash_id,
        t.impressions_month, t.avg_position_month, t.position_tier,
        ROUND(t.ctr_month, 4) AS ctr_month,
        ROUND(ta.tier_avg_ctr, 4) AS tier_avg_ctr,
        ROUND(GREATEST(ta.tier_avg_ctr - t.ctr_month, 0), 4) AS ctr_gap,
        DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_last_update,
        (DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') >= 90) AS is_stale,
        d.word_count
    FROM tiered t
    JOIN tier_avg ta ON t.position_tier = ta.position_tier
    JOIN read_parquet('{rel}/dim_content.parquet') d ON t.content_hash_id = d.content_hash_id
""").df()

queue["baseline_score"] = queue["impressions_month"] * queue["is_stale"].astype(int) * queue["ctr_gap"]
queue["reason_code"] = "stale_ctr_underperformer"
queue["action_label"] = "review_for_refresh"

queue_ranked = queue.sort_values("baseline_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
queue_ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Queue written: {len(queue_ranked):,} rows -> work/outputs/baseline_action_score.csv")
queue_ranked.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Queue written: 101,441 rows -> work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,impressions_month,avg_position_month,position_tier,ctr_month,tier_avg_ctr,ctr_gap,days_since_last_update,is_stale,word_count,baseline_score,reason_code,action_label
0,content_097459d155cccb26,client_20259bd6705d81d4,37930.0,17.550666,3_page_2,0.0015,0.0024,0.0009,124,True,2591,34.1370,stale_ctr_underperformer,review_for_refresh
1,content_9598a57544925111,client_20259bd6705d81d4,24066.0,12.873404,3_page_2,0.0011,0.0024,0.0013,123,True,3966,31.2858,stale_ctr_underperformer,review_for_refresh
2,content_66d1fffc91f4f029,client_20259bd6705d81d4,31038.0,15.216861,3_page_2,0.0017,0.0024,0.0007,124,True,5125,21.7266,stale_ctr_underperformer,review_for_refresh
3,content_b956947c822af734,client_20259bd6705d81d4,22502.0,36.484478,4_deep_20plus,0.0003,0.0012,0.0009,124,True,4860,20.2518,stale_ctr_underperformer,review_for_refresh
4,content_47da45b084a73115,client_20259bd6705d81d4,14774.0,50.756106,4_deep_20plus,0.0001,0.0012,0.0012,124,True,4134,17.7288,stale_ctr_underperformer,review_for_refresh
5,content_ac4e2d9d3bbb06de,client_20259bd6705d81d4,33348.0,23.327797,4_deep_20plus,0.0008,0.0012,0.0005,124,True,4087,16.6740,stale_ctr_underperformer,review_for_refresh
6,content_1d2dc429b857b82c,client_20259bd6705d81d4,6376.0,4.743185,2_striking_4_10,0.0013,0.0032,0.0020,124,True,3752,12.7520,stale_ctr_underperformer,review_for_refresh
7,content_f8e84719d52ab1fe,client_20259bd6705d81d4,3676.0,9.827033,2_striking_4_10,0.0000,0.0032,0.0032,124,True,4152,11.7632,stale_ctr_underperformer,review_for_refresh
8,content_bea86ce3455100b0,client_c182d11e4862a37d,3670.0,6.555793,2_striking_4_10,0.0003,0.0032,0.0030,232,True,1447,11.0100,stale_ctr_underperformer,review_for_refresh
9,content_4981cb34b411063e,client_20259bd6705d81d4,9961.0,12.357374,3_page_2,0.0013,0.0024,0.0011,123,True,4270,10.9571,stale_ctr_underperformer,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = queue_ranked.head(20)[[
    "content_hash_id", "client_hash_id", "baseline_score", "reason_code", "action_label",
    "impressions_month", "position_tier", "ctr_month", "tier_avg_ctr",
    "days_since_last_update", "word_count"
]]
top20

,content_hash_id,client_hash_id,baseline_score,reason_code,action_label,impressions_month,position_tier,ctr_month,tier_avg_ctr,days_since_last_update,word_count
0,content_097459d155cccb26,client_20259bd6705d81d4,34.1370,stale_ctr_underperformer,review_for_refresh,37930.0,3_page_2,0.0015,0.0024,124,2591
1,content_9598a57544925111,client_20259bd6705d81d4,31.2858,stale_ctr_underperformer,review_for_refresh,24066.0,3_page_2,0.0011,0.0024,123,3966
2,content_66d1fffc91f4f029,client_20259bd6705d81d4,21.7266,stale_ctr_underperformer,review_for_refresh,31038.0,3_page_2,0.0017,0.0024,124,5125
3,content_b956947c822af734,client_20259bd6705d81d4,20.2518,stale_ctr_underperformer,review_for_refresh,22502.0,4_deep_20plus,0.0003,0.0012,124,4860
4,content_47da45b084a73115,client_20259bd6705d81d4,17.7288,stale_ctr_underperformer,review_for_refresh,14774.0,4_deep_20plus,0.0001,0.0012,124,4134
5,content_ac4e2d9d3bbb06de,client_20259bd6705d81d4,16.6740,stale_ctr_underperformer,review_for_refresh,33348.0,4_deep_20plus,0.0008,0.0012,124,4087
6,content_1d2dc429b857b82c,client_20259bd6705d81d4,12.7520,stale_ctr_underperformer,review_for_refresh,6376.0,2_striking_4_10,0.0013,0.0032,124,3752
7,content_f8e84719d52ab1fe,client_20259bd6705d81d4,11.7632,stale_ctr_underperformer,review_for_refresh,3676.0,2_striking_4_10,0.0000,0.0032,124,4152
8,content_bea86ce3455100b0,client_c182d11e4862a37d,11.0100,stale_ctr_underperformer,review_for_refresh,3670.0,2_striking_4_10,0.0003,0.0032,232,1447
9,content_4981cb34b411063e,client_20259bd6705d81d4,10.9571,stale_ctr_underperformer,review_for_refresh,9961.0,3_page_2,0.0013,0.0024,123,4270


## Top-20 Review

| # | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | review_for_refresh | Highest score: 37,930 impressions, stale (124d), CTR 0.15% vs tier avg 0.24% | If this is a seasonal dip that self-corrects next month |
| 2 | review_for_refresh | Large gap: CTR 0.11% vs tier avg 0.24%, 24K impressions, stale | If the meta title/description was already updated after March |
| 3 | review_for_refresh | Strong volume (31K impressions) with a real CTR shortfall | If a SERP feature (e.g. featured snippet) is suppressing clicks site-wide, not just here |
| 4 | review_for_refresh | Deep position (36.5) but still meaningful impressions (22.5K) | If position 36 pages structurally never get clicks regardless of CTR fixes |
| 5 | review_for_refresh | Very low CTR (0.01%) at deep position, 14.7K impressions | Same as #4 — deep-position CTR may not be fixable via content edits |
| 6 | review_for_refresh | Solid impressions (33K) at deep position, CTR far below tier avg | If position (not metadata) is the real bottleneck here |
| 7 | review_for_refresh | Smaller volume (6.4K) but a real CTR gap at a decent position (striking) | Lower confidence than top rows due to smaller impression base |
| 8 | review_for_refresh | CTR is literally 0.0000 — zero clicks despite 3,676 impressions | If 0.0000 CTR is a data artifact (e.g. rounding) rather than a real zero |
| 9 | review_for_refresh | Same client cluster, stale (232d), lower volume (3.6K) | First row from a different client — good, but still only 1 of 9 so far |
| 10 | review_for_refresh | Mid-size gap and volume (9.96K), consistent with the pattern above | Same client concentration concern as rows 1-9 |
| 11 | review_for_refresh | Moderate volume (4.9K), same client cluster, smaller gap | Marginal position — could easily be reordered by a small change to the score weights |
| 12 | review_for_refresh | Moderate volume (5.5K), consistent CTR shortfall pattern | Score driven mainly by volume, not by an unusually large gap |
| 13 | review_for_refresh | Second client (`c182d11e...`) appears again — some diversity | Still same content/client cluster as row 9 |
| 14 | review_for_refresh | Smaller score (5.0), volume and gap both modest | Marginal pick — close to the cutoff, low confidence |
| 15 | review_for_refresh | First page from a third distinct client (`157ffe4d...`) | Only one page from this client in top 20 — may be under-represented, not necessarily wrong |
| 16 | review_for_refresh | **word_count is missing (NaN)** — content metadata incomplete | If word_count is missing, I can't confirm this is even a real published article |
| 17 | review_for_refresh | Deep position, low CTR, moderate volume (8.4K) | Score here is fairly low — near the noise floor of the ranking |
| 18 | review_for_refresh | Small gap (0.0008), largely driven by stale flag alone | Weakest CTR-gap justification in the top 20 so far |
| 19 | review_for_refresh | CTR is 0.0000 again, but low volume (1,426) — small evidence base | Zero CTR with under 1,500 impressions could easily be noise |
| 20 | review_for_refresh | Lowest score in the top 20 (2.99): striking-tier position, modest volume (2.1K impressions), smaller CTR gap (0.19% vs 0.32% tier avg) | Borderline pick — sitting right at the cutoff; a slightly different weighting in the score formula could easily drop this row out of the top 20 |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# Look for weak picks: missing metadata, suspiciously low volume, or heavy client concentration
weak_check = queue_ranked.head(20).copy()

# Flag 1: missing word_count
weak_check["flag_missing_wordcount"] = weak_check["word_count"].isna()

# Flag 2: very low impressions (thin evidence for the CTR gap)
weak_check["flag_low_volume"] = weak_check["impressions_month"] < 2000

# Client concentration in the top 20
client_counts = queue_ranked.head(20)["client_hash_id"].value_counts()
print("Client concentration in the top 20:")
print(client_counts)

print("\nRows with missing word_count or very low volume:")
weak_check[weak_check["flag_missing_wordcount"] | weak_check["flag_low_volume"]][
    ["content_hash_id", "client_hash_id", "baseline_score", "impressions_month", "word_count"]
]

Client concentration in the top 20:
client_hash_id
client_20259bd6705d81d4    16
client_c182d11e4862a37d     3
client_157ffe4d4a595515     1
Name: count, dtype: int64

Rows with missing word_count or very low volume:


,content_hash_id,client_hash_id,baseline_score,impressions_month,word_count
15,content_5120dcbbb086843d,client_c182d11e4862a37d,4.5728,1429.0,<NA>
18,content_63701d5ee76253d3,client_20259bd6705d81d4,3.4224,1426.0,3772


In [8]:
# Leakage check: confirm no future-window or label-derived columns went into the score
score_inputs = ["impressions_month", "is_stale", "ctr_gap"]
print("Score built only from:", score_inputs)
print("All three are computed from March 2026 (the feature month) only.")
print("No product decision flags exist in this dataset to leak.")
print("No future window (April+) was used anywhere in the score.")

Score built only from: ['impressions_month', 'is_stale', 'ctr_gap']
All three are computed from March 2026 (the feature month) only.
No product decision flags exist in this dataset to leak.
No future window (April+) was used anywhere in the score.


## Weak Picks + Leakage Check

**Weak picks identified in the top 20:**
- **Row 15** (`content_5120dcbbb086843d`) has a missing `word_count` (NaN) — I can't confirm
  this is a fully-formed published article, which undermines confidence in flagging it for
  refresh.
- **Row 18** (`content_63701d5ee76253d3`) has CTR = 0.0000 with only 1,426 impressions — a
  zero CTR on this small a base is weaker evidence than the same zero on tens of thousands
  of impressions; it could easily be noise rather than a real underperformance.
- **Client concentration:** 16 of the top 20 rows (80%) belong to a single client
  (`client_20259bd6705d81d4`), with only 3 rows from a second client and 1 from a third.
  This means the queue currently over-represents one client's content and under-represents
  everyone else — a real limitation for a queue meant to serve multiple clients fairly. A
  stronger version of this rule would rank within each client separately, or cap how many
  slots one client can take in a shared top-N.

**Leakage check:** the score is built only from `impressions_month`, `is_stale`, and
`ctr_gap` — all computed from the March 2026 feature window alone. No product decision flags
(health_score, priority_score) exist in this warehouse release to leak in. No future window
(April 2026 or later) was used anywhere in scoring or ranking.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.